**carregar arquivos**



In [ ]:
# Importa a biblioteca pandas, usada para leitura e manipulação de tabelas.
import pandas as pd

# Importa a biblioteca NumPy, usada para operações numéricas e matrizes.
import numpy as np


# Importa o ColumnTransformer, que permite aplicar transformações
# diferentes em grupos diferentes de colunas.
from sklearn.compose import ColumnTransformer

# Importa:
# OneHotEncoder → transforma variáveis categóricas em colunas numéricas 0/1.
# StandardScaler → padroniza as variáveis numéricas.
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Importa o TensorFlow para construir a rede neural MLP.
import tensorflow as tf
from tensorflow.keras import layers

# Importa métricas que serão usadas posteriormente para avaliar o modelo:
# accuracy_score → acurácia
# precision_score → precisão
# recall_score → recall/sensibilidade
# f1_score → F1-score
# confusion_matrix → matriz de confusão
# classification_report → relatório com várias métricas
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)


# Carrega o conjunto de treinamento criado e salvo na aula anterior.
train = pd.read_csv('bank_train.csv')

# Carrega o conjunto de validação criado na aula anterior.
validation = pd.read_csv('bank_validation.csv')

# Carrega o conjunto de teste criado na aula anterior.
test = pd.read_csv('bank_test.csv')


# Exibe as dimensões do conjunto de treinamento.
# O resultado possui o formato: (número de amostras, número de colunas).
print('Train:', train.shape)

# Exibe as dimensões do conjunto de validação.
print('Validation:', validation.shape)

# Exibe as dimensões do conjunto de teste.
print('Test:', test.shape)

**separar entrada e alvo**

In [ ]:
# Remove a coluna 'y' do conjunto de treinamento.
# X_train ficará somente com as features usadas como entrada do modelo.
X_train = train.drop(columns=['y'])

# Separa a coluna 'y', que contém a classe verdadeira de cada amostra.
y_train = train['y']


# Remove a coluna 'y' do conjunto de validação.
X_val = validation.drop(columns=['y'])

# Guarda separadamente as classes verdadeiras do conjunto de validação.
y_val = validation['y']


# Remove a coluna 'y' do conjunto de teste.
X_test = test.drop(columns=['y'])

# Guarda separadamente as classes verdadeiras do conjunto de teste.
y_test = test['y']


# Exibe as dimensões da matriz de features de treinamento.
# O formato será: (número de amostras, número de features).
print('X_train:', X_train.shape)

# Exibe as dimensões do vetor com as classes do treinamento.
# Como existe uma classe para cada amostra, teremos apenas uma dimensão.
print('y_train:', y_train.shape)

# Mostra quais classes existem no problema.
# Como o target foi convertido anteriormente para 0 e 1,
# esperamos obter: [0, 1].
print('Classes:', sorted(y_train.unique()))

X_train: (31647, 15)
y_train: (31647,)
Classes: [np.int64(0), np.int64(1)]


**identificar tipos de variáveis**

In [ ]:
# Identifica as colunas numéricas de X_train.
# select_dtypes(include=np.number) seleciona apenas colunas com valores numéricos.
# .columns retorna os nomes dessas colunas.
# .tolist() converte os nomes para uma lista Python.
numericas = X_train.select_dtypes(include=np.number).columns.tolist()

# Identifica as colunas categóricas de X_train.
# exclude=np.number seleciona todas as colunas que não são numéricas.
categoricas = X_train.select_dtypes(exclude=np.number).columns.tolist()


# Exibe o título da lista de variáveis numéricas.
print('Numéricas:')

# Exibe os nomes das colunas numéricas.
print(numericas)


# Pula uma linha e exibe o título da lista de variáveis categóricas.
print('\nCategóricas:')

# Exibe os nomes das colunas categóricas.
print(categoricas)

Numéricas:
['age', 'balance', 'day_of_week', 'campaign', 'pdays', 'previous']

Categóricas:
['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']


**criar o pré-processamento**

In [ ]:
# Tenta criar o OneHotEncoder usando o parâmetro mais recente
# do scikit-learn: sparse_output=False.
# Isso faz com que a saída seja uma matriz NumPy comum,
# e não uma matriz esparsa.
try:
    encoder = OneHotEncoder(
        handle_unknown='ignore',
        sparse_output=False
    )

# Em versões mais antigas do scikit-learn,
# o parâmetro sparse_output ainda não existe.
# Nesse caso, usamos sparse=False.
except TypeError:
    encoder = OneHotEncoder(
        handle_unknown='ignore',
        sparse=False
    )


# Cria o pré-processamento das features.
# O ColumnTransformer permite aplicar uma transformação
# diferente para cada grupo de colunas.
preprocess = ColumnTransformer([

    # Aplica StandardScaler às variáveis numéricas.
    # O scaler aprende média e desvio-padrão usando o Train.
    ('num', StandardScaler(), numericas),

    # Aplica One-Hot Encoding às variáveis categóricas.
    ('cat', encoder, categoricas)
])


# Aprende as transformações usando APENAS o conjunto de treinamento
# e, em seguida, transforma X_train.
#
# Aqui o StandardScaler aprende média e desvio-padrão
# e o OneHotEncoder aprende quais categorias existem.
X_train_prep = preprocess.fit_transform(X_train)


# Aplica em Validation exatamente as mesmas transformações
# aprendidas anteriormente com o Train.
#
# Não usamos fit_transform aqui para evitar data leakage.
X_val_prep = preprocess.transform(X_val)


# Aplica também ao Test as transformações aprendidas no Train.
#
# O conjunto de teste não participa do aprendizado
# do pré-processamento.
X_test_prep = preprocess.transform(X_test)


# Exibe o shape do conjunto de treinamento depois do pré-processamento.
print(X_train_prep.shape)

# Exibe o shape do conjunto de validação depois do pré-processamento.
print(X_val_prep.shape)

(31647, 50)
(6782, 50)


**treinar o MLP**


In [ ]:
# Cria o modelo MLP.
modelo = tf.keras.Sequential([

    # Define o número de entradas da rede.
    layers.Input(shape=(X_train_prep.shape[1],)),

    # Primeira camada oculta com 32 neurônios e ativação ReLU.
    layers.Dense(32, activation='relu'),

    # Segunda camada oculta com 16 neurônios e ativação ReLU.
    layers.Dense(16, activation='relu'),

    # Camada de saída para classificação binária.
    layers.Dense(1, activation='sigmoid')
])


# Define como o modelo será treinado.
modelo.compile(

    # Adam será usado para atualizar os pesos da rede.
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),

    # Função de perda adequada para classificação binária.
    loss='binary_crossentropy',

    # Métrica acompanhada durante o treinamento.
    metrics=['accuracy']
)


# Treina o MLP usando o conjunto de treinamento
# e acompanha o desempenho no conjunto de validação.
history = modelo.fit(
    X_train_prep,
    y_train,
    validation_data=(X_val_prep, y_val),
    epochs=20,
    batch_size=32
)


# Exibe a arquitetura da rede e a quantidade de parâmetros.
modelo.summary()


**avaliar na validação**

In [ ]:
# Usa o MLP treinado para calcular a probabilidade
# das amostras do conjunto de validação pertencerem à classe 1.
y_val_prob = modelo.predict(X_val_prep).ravel()

# Converte as probabilidades em classes:
# probabilidade >= 0.5 → classe 1
# probabilidade < 0.5  → classe 0
y_val_pred = (y_val_prob >= 0.5).astype(int)


# Calcula a acurácia:
# proporção total de previsões corretas.
print('Accuracy:', accuracy_score(y_val, y_val_pred))


# Calcula a precisão:
# entre os casos previstos como classe 1,
# quantos realmente pertencem à classe 1.
#
# zero_division=0 evita erro caso o modelo
# não faça nenhuma previsão positiva.
print(
    'Precision:',
    precision_score(y_val, y_val_pred, zero_division=0)
)


# Calcula o recall:
# entre todos os casos que realmente são classe 1,
# quantos o modelo conseguiu identificar.
print(
    'Recall:',
    recall_score(y_val, y_val_pred, zero_division=0)
)


# Calcula o F1-score:
# combina Precision e Recall em uma única métrica.
print(
    'F1:',
    f1_score(y_val, y_val_pred, zero_division=0)
)


# Exibe um relatório mais completo de classificação.
print('\nRelatório:')

# O classification_report apresenta, para cada classe:
# Precision
# Recall
# F1-score
# Support = número de amostras daquela classe.
print(
    classification_report(
        y_val,
        y_val_pred,
        zero_division=0
    )
)


**matriz de confusão**

In [ ]:
# Calcula a matriz de confusão comparando:
# y_val      → classes verdadeiras
# y_val_pred → classes previstas pelo MLP
cm = confusion_matrix(y_val, y_val_pred)

# Exibe a matriz de confusão.
print(cm)


# A matriz é organizada assim:
#
# [[TN, FP],
#  [FN, TP]]
#
# TN = True Negative  → verdadeiro negativo
# FP = False Positive → falso positivo
# FN = False Negative → falso negativo
# TP = True Positive  → verdadeiro positivo

**testar apenas no final**

In [ ]:
# Usa o MLP já treinado para calcular a probabilidade
# das amostras do conjunto de teste pertencerem à classe 1.
y_test_prob = modelo.predict(X_test_prep).ravel()

# Converte as probabilidades em classes.
y_test_pred = (y_test_prob >= 0.5).astype(int)


# Exibe o relatório de classificação no conjunto de teste.
#
# O relatório mostra, para cada classe:
# precision  → entre os casos previstos como aquela classe,
#              quantos estavam corretos
#
# recall     → entre os casos realmente daquela classe,
#              quantos foram identificados
#
# f1-score   → equilíbrio entre precision e recall
#
# support    → quantidade real de amostras daquela classe
print(
    classification_report(
        y_test,
        y_test_pred,
        zero_division=0
    )
)


# Calcula e exibe a matriz de confusão do conjunto de teste.
#
# Organização:
#
# [[TN, FP],
#  [FN, TP]]
#
# TN → verdadeiro negativo
# FP → falso positivo
# FN → falso negativo
# TP → verdadeiro positivo
print(
    confusion_matrix(
        y_test,
        y_test_pred
    )
)
